In [1]:
import pandas as pd
import numpy as np

from joblib import dump
import os

import matplotlib.pyplot as plt
import seaborn as sns
import klib

from sklearn.model_selection import train_test_split, KFold,cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
labeller = LabelEncoder()

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression

RF_Reg = RandomForestRegressor()
DT_Clas = DecisionTreeClassifier()
svc = SVC()
LR_Reg = LinearRegression()

from sklearn.metrics import classification_report,accuracy_score

In [2]:
df = pd.read_csv(r"C:\Users\bunyo\OneDrive\Desktop\git_dataset\car_price_prediction.csv")

In [3]:
df.tail()

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
19232,45798355,8467,-,MERCEDES-BENZ,CLK 200,1999,Coupe,Yes,CNG,2.0 Turbo,300000 km,4.0,Manual,Rear,02-Mar,Left wheel,Silver,5
19233,45778856,15681,831,HYUNDAI,Sonata,2011,Sedan,Yes,Petrol,2.4,161600 km,4.0,Tiptronic,Front,04-May,Left wheel,Red,8
19234,45804997,26108,836,HYUNDAI,Tucson,2010,Jeep,Yes,Diesel,2,116365 km,4.0,Automatic,Front,04-May,Left wheel,Grey,4
19235,45793526,5331,1288,CHEVROLET,Captiva,2007,Jeep,Yes,Diesel,2,51258 km,4.0,Automatic,Front,04-May,Left wheel,Black,4
19236,45813273,470,753,HYUNDAI,Sonata,2012,Sedan,Yes,Hybrid,2.4,186923 km,4.0,Automatic,Front,04-May,Left wheel,White,12


In [4]:
df.drop(columns=['ID'],inplace=True)

In [47]:
class Preprocessing:
    def __init__(self,df):
        self.df=df
        
# Encoding qiluvchi
    def encoding_qilish(self):
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                if self.df[col].nunique() <= 2:
                    new_df = pd.get_dummies(self.df[col], prefix='New', dtype=int)
                    self.df.drop(columns=[col],inplace=True)
                    self.df = pd.concat([self.df,new_df],axis=1)
                else:
                    self.df[col] = labeller.fit_transform(self.df[col])
        return self
    
# Scale qiluvchi
    def scaling_qilish(self):
        for cols in self.df.columns:
            if self.df[cols].dtype != 'object' and self.df[cols].name != 'Price':
                self.df[cols] = minmax.fit_transform(self.df[[cols]])
        return self
    
# To'ldruvchi
    def fillingNan(self):
        for cols in self.df.columns:
            if self.df[cols].isnull().any():
                if self.df[cols].dtype == 'object':
                    self.df[cols].fillna(self.df[cols].mode()[0], inplace=True)
                else:
                    self.df[cols].fillna(self.df[cols].mean(),inplace=True)
        return self          
    
# Skewness ni tog'irlash
    # def skw(self):
    #     for cols in self.df.columns:
    #         if self.df[cols].dtype != 'object':
    #             self.df[cols] = np.log()


In [24]:
dt = Preprocessing(df)
dt.fillingNan().encoding_qilish().scaling_qilish()
df1 = dt.df

In [7]:
skeww = df1.drop('Price',axis=1).skew()

In [8]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Price                 19237 non-null  int64  
 1   Levy                  19237 non-null  float64
 2   Manufacturer          19237 non-null  float64
 3   Model                 19237 non-null  float64
 4   Prod. year            19237 non-null  float64
 5   Category              19237 non-null  float64
 6   Fuel type             19237 non-null  float64
 7   Engine volume         19237 non-null  float64
 8   Mileage               19237 non-null  float64
 9   Cylinders             19237 non-null  float64
 10  Gear box type         19237 non-null  float64
 11  Drive wheels          19237 non-null  float64
 12  Doors                 19237 non-null  float64
 13  Color                 19237 non-null  float64
 14  Airbags               19237 non-null  float64
 15  New_No             

In [9]:
skeww

Levy                    0.075949
Manufacturer            0.162284
Model                  -0.061702
Prod. year             -2.082261
Category               -0.165874
Fuel type              -0.421426
Engine volume           1.073482
Mileage                 0.241975
Cylinders               2.091083
Gear box type           1.369430
Drive wheels           -0.012095
Doors                  -2.953955
Color                  -0.163907
Airbags                 0.082012
New_No                  1.009981
New_Yes                -1.009981
New_Left wheel         -3.169873
New_Right-hand drive    3.169873
dtype: float64

# df ajratamiz

In [10]:
x = df1.drop('Price', axis=1)
y = df1['Price']

# log ni qollaymiz

In [11]:
log_transformation=skeww[(skeww>=0.5)].index.tolist()
print(log_transformation)

['Engine volume', 'Cylinders', 'Gear box type', 'New_No', 'New_Right-hand drive']


# yangi ustun yaratamiz

In [12]:
for col in log_transformation:
    x['log_'+col] = np.log1p(x[col])

# train test split qilamiz

In [13]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [14]:
from sklearn.metrics import r2_score,mean_absolute_error

In [15]:
RF_Reg.fit(x_train,y_train)

RandomForestRegressor()

In [16]:
pred1 = RF_Reg.predict(x_test)
print(r2_score(y_test,pred1))
print(mean_absolute_error(y_test,pred1))

-275.41205086959036
10760.748878494984


In [17]:
LR_Reg.fit(x_train,y_train)

LinearRegression()

In [18]:
pred2 = LR_Reg.predict(x_test)
print(r2_score(y_test,pred2))
print(mean_absolute_error(y_test,pred2))

-0.1489650930123112
13334.680215924785


# log siz bajaramiz

In [48]:
df = pd.read_csv(r"C:\Users\bunyo\OneDrive\Desktop\git_dataset\car_price_prediction.csv")

In [49]:
dt1 = Preprocessing(df)
dt1.fillingNan().encoding_qilish().scaling_qilish()
preData = dt1.df

In [50]:
preData

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Color,Airbags,New_No,New_Yes,New_Left wheel,New_Right-hand drive
0,0.993528,13328,0.204301,0.500000,0.781624,0.876543,0.4,0.333333,0.594340,0.369243,0.333333,0.000000,0.0,0.5,0.800000,0.7500,0.0,1.0,1.0,0.0
1,0.956715,16621,0.007168,0.125000,0.414097,0.888889,0.4,0.833333,0.528302,0.385116,0.333333,0.666667,0.0,0.5,0.066667,0.5000,1.0,0.0,1.0,0.0
2,0.998315,8467,0.000000,0.328125,0.430459,0.827160,0.3,0.833333,0.207547,0.408535,0.200000,1.000000,0.5,0.5,0.066667,0.1250,1.0,0.0,0.0,1.0
3,0.998107,3607,0.917563,0.250000,0.415985,0.888889,0.4,0.333333,0.433962,0.313947,0.200000,0.000000,0.0,0.5,0.933333,0.0000,0.0,1.0,1.0,0.0
4,0.999705,11726,0.655914,0.328125,0.430459,0.925926,0.3,0.833333,0.207547,0.962269,0.200000,0.000000,0.5,0.5,0.800000,0.2500,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19232,0.999270,8467,0.000000,0.562500,0.242291,0.740741,0.1,0.000000,0.349057,0.571689,0.200000,0.333333,1.0,0.0,0.800000,0.3125,0.0,1.0,1.0,0.0
19233,0.998492,15681,0.903226,0.359375,0.839522,0.888889,0.9,0.833333,0.415094,0.288056,0.200000,0.666667,0.5,0.5,0.733333,0.5000,0.0,1.0,1.0,0.0
19234,0.999535,26108,0.910394,0.359375,0.907489,0.876543,0.4,0.166667,0.339623,0.083008,0.200000,0.000000,0.5,0.5,0.466667,0.2500,0.0,1.0,1.0,0.0
19235,0.999077,5331,0.148746,0.125000,0.286973,0.839506,0.4,0.166667,0.339623,0.730159,0.200000,0.000000,0.5,0.5,0.066667,0.2500,0.0,1.0,1.0,0.0


In [51]:
x = preData.drop('Price',axis=1)
y = preData['Price']

In [52]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# Random Forest

In [53]:
RF_Reg.fit(x_train,y_train)

RandomForestRegressor()

In [ ]:
pred2 = RF_Reg.predict(x_test)
print(r2_score(y_test,pred2))

print(mean_absolute_error(y_test,pred2))

-255.15768394065037
10197.753487525988


# Tabulate

In [55]:
from tabulate import tabulate

In [56]:
head = ['Model Name','r2 score(with log)', 'MAE(with log)','orginal r2 score','MAE']

data = [['Random forest','-0.1489','13334.68','-255.15','10197.75']]

data1 = tabulate(data,headers=head,tablefmt='fancy_grid')

print(data1)

╒═══════════════╤══════════════════════╤═════════════════╤════════════════════╤═════════╕
│ Model Name    │   r2 score(with log) │   MAE(with log) │   orginal r2 score │     MAE │
╞═══════════════╪══════════════════════╪═════════════════╪════════════════════╪═════════╡
│ Random forest │              -0.1489 │         13334.7 │            -255.15 │ 10197.8 │
╘═══════════════╧══════════════════════╧═════════════════╧════════════════════╧═════════╛
